# RAG completo con Elasticsearch y LangChain 1.x

Este notebook practica un flujo completo de **Retrieval Augmented Generation (RAG)** usando un PDF sobre redes neuronales como fuente documental.

La práctica está dividida en tres partes:

1. **Ingesta de datos:** carga de PDF, fragmentación con `RecursiveCharacterTextSplitter`, embeddings y almacenamiento en Elasticsearch.
2. **Agente básico con RAG:** creación de un agente en LangChain 1.x usando una herramienta propia que envuelve el `vector_store` y permite filtros por metadata.
3. **Búsquedas avanzadas:** filtros por metadata, filtros por varios campos, BM25, búsqueda híbrida BM25 + vector, búsqueda híbrida + metadata, reranking y respuestas con sustento de documento y página.

> Importante: el objetivo didáctico es mostrar cómo la metadata permite pasar de una búsqueda semántica simple a una recuperación más controlada, explicable y auditable.

# Instalación de dependencias


In [1]:
%pip install -q "langchain>=1.0" langchain-google-genai langchain-elasticsearch langchain-community langchain-text-splitters pypdf elasticsearch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# 1. Configuración de credenciales y variables


- `/content/api_key.txt` para `GEMINI_API_KEY`
- `/content/elasticstore_urp.txt` para la contraseña de Elasticsearch

También puedes usar variables de entorno.

In [2]:
import os
import getpass
from pathlib import Path


def read_secret_file(path: str) -> str | None:
    file_path = Path(path)
    if file_path.exists():
        return file_path.read_text(encoding="utf-8").strip()
    return None

# Gemini
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or read_secret_file("/content/api_key.txt")
if not gemini_key:
    gemini_key = getpass.getpass("Ingresa tu GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["GOOGLE_API_KEY"] = gemini_key

# Elasticsearch
ES_URL = os.getenv("ES_URL", "http://104.198.172.31:9200")
ES_USER = os.getenv("ES_USER", "elastic")
ES_PASSWORD = os.getenv("ES_PASSWORD") or read_secret_file("/content/elasticstore_urp.txt")
if not ES_PASSWORD:
    ES_PASSWORD = getpass.getpass("Ingresa la contraseña de Elasticsearch: ")

INDEX_NAME = os.getenv("ES_INDEX", "rag_hiraoka_cambiosydevoluciones")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-2")
CHAT_MODEL = os.getenv("CHAT_MODEL", "google_genai:gemini-2.5-flash")

print("Configuración cargada")
print("ES_URL:", ES_URL)
print("INDEX_NAME:", INDEX_NAME)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("CHAT_MODEL:", CHAT_MODEL)

Configuración cargada
ES_URL: http://104.198.172.31:9200
INDEX_NAME: rag_hiraoka_terminosycondiciones
EMBEDDING_MODEL: gemini-embedding-2
CHAT_MODEL: google_genai:gemini-2.5-flash


# Parte 1 — Ingesta de datos

En esta parte cargaremos el PDF, construiremos metadata útil para búsquedas filtradas y guardaremos los chunks en Elasticsearch.

La metadata se construye considerando que el PDF trata sobre **redes neuronales**, con secciones como introducción, reseña histórica, generalidades, elementos básicos, aprendizaje, topologías, aplicaciones y software comercial.

## 1.1 Cargar el PDF en Colab

El notebook espera el archivo `matich-redesneuronales.pdf`. Si no existe en `/content`, se abrirá un uploader para cargarlo manualmente.

In [3]:
PDF_PATH = Path("/content/CambiosyDevoluciones.pdf")

if not PDF_PATH.exists():
    try:
        from google.colab import files
        print("Sube el archivo Terminos_Condiciones.pdf")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(first_name).rename(PDF_PATH)
            print(f"PDF cargado como: {PDF_PATH}")
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró /content/CambiosyDevoluciones.pdf. "
            "Sube el PDF manualmente o cambia PDF_PATH."
        ) from e

print("PDF listo:", PDF_PATH)

PDF listo: /content/CambiosyDevoluciones.pdf


## 1.2 Loader para PDF

Usaremos `PyPDFLoader` para extraer el texto por página. Cada página se convierte en un `Document` de LangChain.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")
print("Metadata base:", pages[0].metadata)

/tmp/ipykernel_686/3086807236.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Páginas cargadas: 3
Metadata base: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0', 'creationdate': '2026-06-18T15:16:08+00:00', 'title': '▷ Cambios y Devoluciones en Hiraoka.com.pe', 'moddate': '2026-06-18T15:16:08+00:00', 'source': '/content/CambiosyDevoluciones.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


## 1.3 Enriquecimiento de metadata

La metadata permitirá filtrar por:

- `documento`
- `page_number`
- `section_slug`
- `section_title`
- `temas`
- `tipo_contenido`
- `dominio`
- `autor`
- `anio`

Esto es importante porque luego la herramienta del agente podrá buscar por tema, página, sección o tipo de contenido.

### Paso 1 - Crear secciones reales del documento

In [5]:
import re
import unicodedata
from dataclasses import dataclass
from typing import Optional

DOCUMENT_TITLE = "Cambios y Devoluciones"
DOCUMENT_AUTHOR = "Hiraoka"
DOCUMENT_YEAR = 2026
DOCUMENT_DOMAIN = "Cambios y Devoluciones"


def normalize_text(text: str) -> str:
    return (
        unicodedata.normalize("NFKD", text.lower())
        .encode("ascii", "ignore")
        .decode("ascii")
    )


@dataclass
class Section:
    slug: str
    title: str
    parent: Optional[str]
    page_start: int
    page_end: int
    text: str

SECTION_TITLES = [

    # Documento
    {   "slug": "politica_cambios_devoluciones_garantias", "title": "POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS"},

    # I
    {
        "slug": "politica_devoluciones",
        "title": "I. Política de devoluciones brindadas por Hiraoka",
        "parent": "politica_cambios_devoluciones_garantias"
    },

    {
        "slug": "condiciones_devolucion",
        "title": "Condiciones para solicitar reparación, restitución o devolución",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "productos_usados",
        "title": "Productos usados o con desgaste",
        "parent": "condiciones_devolucion"
    },

    {
        "slug": "acreditacion_titular",
        "title": "Acreditación del titular de la compra",
        "parent": "condiciones_devolucion"
    },

    {
        "slug": "productos_digitales",
        "title": "Restricción para productos digitales, software y licencias",
        "parent": "condiciones_devolucion"
    },

    {
        "slug": "limites_codigo_consumidor",
        "title": "Limitaciones conforme al Código del Consumidor",
        "parent": "condiciones_devolucion"
    },

    {
        "slug": "derecho_restitucion",
        "title": "Derecho de restitución",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "solicitud_devolucion",
        "title": "Procedimiento para solicitar reparación, restitución o devolución",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "devolucion_dinero",
        "title": "Modalidades de devolución de dinero",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "garantias_hiraoka",
        "title": "Garantías brindadas por Hiraoka",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "derecho_arrepentimiento",
        "title": "Derecho de arrepentimiento",
        "parent": "politica_devoluciones"
    },

    {
        "slug": "condiciones_arrepentimiento",
        "title": "Condiciones para devolución por arrepentimiento",
        "parent": "derecho_arrepentimiento"
    },

    # II

    {
        "slug": "garantias_fabricantes",
        "title": "II. Política de garantías de los Fabricantes/Proveedores",
        "parent": "politica_cambios_devoluciones_garantias"
    },

    {
        "slug": "garantia_fabricante",
        "title": "Garantías otorgadas por Fabricantes o Proveedores",
        "parent": "garantias_fabricantes"
    },

    {
        "slug": "intermediacion_hiraoka",
        "title": "Intermediación de Hiraoka en garantías",
        "parent": "garantias_fabricantes"
    },

    {
        "slug": "devolucion_garantia",
        "title": "Devolución de dinero por garantía",
        "parent": "garantias_fabricantes"
    },

    {
        "slug": "plazos_garantia",
        "title": "Plazos para notas de crédito, cheques y extornos",
        "parent": "garantias_fabricantes"
    },

    # III

    {
        "slug": "disposiciones_generales",
        "title": "III. Disposiciones Generales",
        "parent": "politica_cambios_devoluciones_garantias"
    },

    {
        "slug": "no_reconocimiento_responsabilidad",
        "title": "La restitución no implica aceptación de responsabilidad",
        "parent": "disposiciones_generales"
    },

    {
        "slug": "limite_monto_devolucion",
        "title": "Límite del monto de devolución",
        "parent": "disposiciones_generales"
    },

    {
        "slug": "conflicto_solucionado",
        "title": "Conformidad del cliente con la solución aplicada",
        "parent": "disposiciones_generales"
    },

    # IV

    {
        "slug": "productos_sensibles",
        "title": "IV. Cambios y devoluciones de productos sensibles",
        "parent": "politica_cambios_devoluciones_garantias"
    },

    {
        "slug": "definicion_productos_sensibles",
        "title": "Definición de productos sensibles",
        "parent": "productos_sensibles"
    },

    {
        "slug": "restricciones_productos_sensibles",
        "title": "Restricciones para cambios y devoluciones",
        "parent": "productos_sensibles"
    },

    {
        "slug": "inspeccion_productos_sensibles",
        "title": "Inspección y evaluación del producto",
        "parent": "productos_sensibles"
    },

    {
        "slug": "consultas_productos",
        "title": "Consultas sobre características de productos",
        "parent": "productos_sensibles"
    },

    {
        "slug": "cambios_por_color_diseno",
        "title": "Restricciones por color, diseño o modo de uso",
        "parent": "productos_sensibles"
    }
]

### Paso 2 - Encontrar dónde empieza cada sección

In [6]:
def find_section_starts(pages):
    full_text = ""
    char_to_page = []

    for idx, page in enumerate(pages):
        start = len(full_text)
        full_text += page.page_content + "\n\n"
        end = len(full_text)
        char_to_page.append((start, end, idx + 1))

    def get_page_number(char_idx):
        for start, end, p_num in char_to_page:
            if start <= char_idx < end:
                return p_num
        return len(pages)

    norm_full = normalize_text(full_text)
    detected = []

    for sec in SECTION_TITLES:
        norm_title = normalize_text(sec["title"])
        pos = norm_full.find(norm_title)
        if pos != -1:
            detected.append({
                "slug": sec["slug"],
                "title": sec["title"],
                "parent": sec.get("parent"),
                "char_start": pos,
                "page": get_page_number(pos)
            })

    detected.sort(key=lambda x: x["char_start"])
    return detected, full_text, get_page_number

### Paso 3 - Construir objetos Section

In [7]:
def build_sections(pages):
    starts, full_text, get_page_number = find_section_starts(pages)
    sections = []

    for i, section in enumerate(starts):
        start_pos = section["char_start"]
        end_pos = starts[i+1]["char_start"] if i < len(starts)-1 else len(full_text)

        # Extraer el texto exacto entre esta sección y la siguiente
        section_text = full_text[start_pos:end_pos].strip()
        page_start = section["page"]
        page_end = get_page_number(end_pos - 1)

        sections.append(
            Section(
                slug=section["slug"],
                title=section["title"],
                parent=section["parent"],
                page_start=page_start,
                page_end=page_end,
                text=section_text
            )
        )
    return sections

### Ejecutar

In [8]:
sections = build_sections(pages)

print(f"Secciones encontradas: {len(sections)}")

for s in sections:
    print(
        s.slug,
        s.page_start,
        s.page_end
    )

Secciones encontradas: 7
politica_cambios_devoluciones_garantias 1 1
politica_devoluciones 1 1
derecho_restitucion 1 1
derecho_arrepentimiento 1 2
garantias_fabricantes 2 2
disposiciones_generales 2 2
productos_sensibles 2 3


## 1.4 Fragmentación con RecursiveCharacterTextSplitter

Fragmentaremos por página para preservar la trazabilidad. Cada chunk hereda la metadata de su página original.

In [9]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=250
)

### Convertimos las secciones a documentos:

In [10]:
section_docs = []

# Mapeamos los títulos de secciones por su slug para resolver la jerarquía
sections_by_slug = {s["slug"]: s for s in SECTION_TITLES}

for section in sections:
    metadata = {
        "documento": DOCUMENT_TITLE,
        "autor": DOCUMENT_AUTHOR,
        "anio": DOCUMENT_YEAR,
        "dominio": DOCUMENT_DOMAIN,
        "page_start": section.page_start,
        "page_end": section.page_end
    }

    # Determinar jerarquía
    parent_slug = section.parent
    if not parent_slug:
        # Es sección principal (Main)
        metadata["section_slug"] = section.slug
        metadata["section_title"] = section.title
    else:
        grandparent_slug = sections_by_slug[parent_slug].get("parent")
        if not grandparent_slug:
            # Es subsección (Sub)
            metadata["section_slug"] = parent_slug
            metadata["section_title"] = sections_by_slug[parent_slug]["title"]
            metadata["subsection_slug"] = section.slug
            metadata["subsection_title"] = section.title
        else:
            # Es sub-subsección (Sub-sub)
            metadata["section_slug"] = grandparent_slug
            metadata["section_title"] = sections_by_slug[grandparent_slug]["title"]
            metadata["subsection_slug"] = parent_slug
            metadata["subsection_title"] = sections_by_slug[parent_slug]["title"]
            metadata["sub_subsection_slug"] = section.slug
            metadata["sub_subsection_title"] = section.title

    doc = Document(
        page_content=section.text,
        metadata=metadata
    )
    section_docs.append(doc)

In [11]:
splits = splitter.split_documents(
    section_docs
)

In [12]:
print(splits[0].page_content[:900])

POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS


### Paso 5 - Metadata a nivel chunk

In [13]:
for idx, chunk in enumerate(splits):
    specific_slug = (
        chunk.metadata.get("sub_subsection_slug")
        or chunk.metadata.get("subsection_slug")
        or chunk.metadata.get("section_slug")
    )
    chunk.metadata["chunk_id"] = f"{specific_slug}_{idx:04d}"
    chunk.metadata["chunk_number"] = idx

In [ ]:
print(splits[2].metadata)

{'documento': 'Terminos y Condiciones', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Términos y Condiciones', 'page_start': 1, 'page_end': 1, 'section_slug': 'uso_portal', 'section_title': 'Uso del Portal', 'chunk_id': 'uso_portal_0002', 'chunk_number': 2}


## 1.5 Resumen de metadata generada

Antes de indexar, revisamos la distribución de chunks por sección y tipo de contenido.

In [14]:
import pandas as pd

metadata_df = pd.DataFrame([doc.metadata for doc in splits])

resumen_secciones = (
    metadata_df.groupby(["section_title", "page_start"])
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

resumen_tipo = (
    metadata_df.groupby("section_title")
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

print("Chunks por sección")
display(resumen_secciones)
print("Chunks por tipo de contenido")
display(resumen_tipo)

Chunks por sección


,section_title,page_start,chunks
0,"POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS",1,6
1,"POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS",2,6


Chunks por tipo de contenido


,section_title,chunks
0,"POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS",12


## 1.6 Embeddings y conexión a Elasticsearch

Conectamos a `ElasticsearchStore` y guardaremos los chunks con IDs estables basados en documento, página y número de chunk.

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = ElasticsearchStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
)

# Cambia a False si quieres reutilizar el índice ya creado.
RECREATE_INDEX = True

if RECREATE_INDEX and vector_store.client.indices.exists(index=INDEX_NAME):
    vector_store.client.indices.delete(index=INDEX_NAME)
    print(f"Índice eliminado: {INDEX_NAME}")

ids = [doc.metadata["chunk_id"] for doc in splits]

inserted_ids = vector_store.add_documents(
    documents=splits,
    ids=ids,
    bulk_kwargs={"chunk_size": 100},
)

vector_store.client.indices.refresh(index=INDEX_NAME)
print(f"Chunks indexados: {len(inserted_ids)}")

Índice eliminado: rag_hiraoka_terminosycondiciones
Chunks indexados: 12


## 1.7 Validación rápida de recuperación vectorial

Probamos una búsqueda semántica simple antes de construir el agente.

In [16]:
from collections import Counter


def score_label(score):

    if score is None:
        return ""

    if score >= 0.90:
        return "🟢 Excelente"

    if score >= 0.80:
        return "🟡 Bueno"

    return "🔴 Débil"


def print_retrieval_results(results, max_chars=600):

    print("\n" + "=" * 120)
    print("RESULTADOS DE RECUPERACIÓN")
    print("=" * 120)

    section_counter = Counter()

    for i, item in enumerate(results, start=1):

        # Compatible con similarity_search()
        # y similarity_search_with_score()

        if isinstance(item, tuple):
            doc, score = item
        else:
            doc, score = item, None

        meta = doc.metadata

        section_slug = meta.get("section_slug", "N/A")
        section_counter[section_slug] += 1

        print("\n" + "=" * 120)

        header = f"Resultado {i}"

        if score is not None:
            header += (
                f" | score={score:.4f}"
                f" ({score_label(score)})"
            )

        print(header)

        print("-" * 120)

        print(
            f"Chunk ID      : "
            f"{meta.get('chunk_id')}"
        )

        print(
            f"Documento     : "
            f"{meta.get('documento')}"
        )

        print(
            f"Sección       : "
            f"{meta.get('section_title')} "
            f"({meta.get('section_slug')})"
        )

        print(
            f"Páginas       : "
            f"{meta.get('page_start')} "
            f"- "
            f"{meta.get('page_end')}"
        )

        print(
            f"Autor         : "
            f"{meta.get('autor')}"
        )

        print(
            f"Año           : "
            f"{meta.get('anio')}"
        )

        print(
            f"Dominio       : "
            f"{meta.get('dominio')}"
        )

        print(
            f"Categoría SAC : "
            f"{meta.get('sac_category', 'N/A')}"
        )

        print(
            f"Tipo Cláusula : "
            f"{meta.get('clause_type', 'N/A')}"
        )

        print(
            f"Nivel Riesgo  : "
            f"{meta.get('risk_level', 'N/A')}"
        )

        entities = meta.get("entities", [])

        print(
            f"Entidades     : "
            f"{', '.join(entities) if entities else 'N/A'}"
        )

        print("-" * 120)

        preview = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        print(preview)

    print("\n" + "=" * 120)
    print("DISTRIBUCIÓN POR SECCIONES")
    print("=" * 120)

    for section, qty in section_counter.items():
        print(f"{section:<30} {qty}")

In [17]:
query = "Cual es el proceso para la devolución"

results = vector_store.similarity_search_with_score(
    query,
    k=5
)

print_retrieval_results(results)


RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8704 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : politica_cambios_devoluciones_garantias_0000
Documento     : Cambios y Devoluciones
Sección       : POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS (politica_cambios_devoluciones_garantias)
Páginas       : 1 - 1
Autor         : Hiraoka
Año           : 2026
Dominio       : Cambios y Devoluciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS

Resultado 2 | score=0.8643 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : derecho_restitucion_0004
Documento     : Cambios y Devoluciones
Sección       : P

# Parte 2 — Agente básico con RAG mediante herramienta propia

En esta parte construiremos un agente con `create_agent`, pero la recuperación no se expondrá con `retriever.as_tool()` ni con `chain.as_tool()`.

Crearemos una herramienta propia con `@tool` que envuelve el `vector_store` y permite filtros por metadata. Este enfoque permite controlar:

- `section_slug`
- rango de páginas
- `tipo_contenido`
- `temas`
- cantidad de resultados
- formato de salida con fuentes

## 2.1 Constructor de filtros por metadata

Elasticsearch permite filtrar usando expresiones como `term`, `terms` y `range`. La metadata quedó guardada dentro del objeto `metadata`, por eso los filtros usan campos como `metadata.section_slug.keyword` o `metadata.page_number`.

In [27]:
from typing import Optional


def build_metadata_filter(
    section_slug: Optional[str] = None,
    documento: Optional[str] = None,
) -> list[dict]:
    """
    Construye filtros Elasticsearch para búsqueda híbrida RAG.

    Metadata soportada actualmente:
    - documento
    - section_slug (soporta buscar en cualquier nivel de la jerarquía: section, subsection, sub_subsection)
    """

    filters = []

    if documento:
        filters.append({
            "term": {
                "metadata.documento.keyword": documento
            }
        })

    if section_slug:
        filters.append({
            "bool": {
                "should": [
                    {"term": {"metadata.section_slug.keyword": section_slug}},
                    {"term": {"metadata.subsection_slug.keyword": section_slug}},
                    {"term": {"metadata.sub_subsection_slug.keyword": section_slug}}
                ],
                "minimum_should_match": 1
            }
        })

    return filters

In [19]:
filters = build_metadata_filter(
    documento="CambiosyDevoluciones",
    section_slug="devolucion_dinero"
)

print(filters)

[{'term': {'metadata.documento.keyword': 'CambiosyDevoluciones'}}, {'bool': {'should': [{'term': {'metadata.section_slug.keyword': 'devolucion_dinero'}}, {'term': {'metadata.subsection_slug.keyword': 'devolucion_dinero'}}, {'term': {'metadata.sub_subsection_slug.keyword': 'devolucion_dinero'}}], 'minimum_should_match': 1}}]


## 2.2 Herramienta RAG personalizada

La herramienta recibe parámetros semánticos y estructurados. Internamente construye filtros de Elasticsearch y ejecuta la búsqueda sobre el vector store.

In [28]:
from typing import Optional

from pydantic import BaseModel, Field


class BuscarTerminosHiraokaInput(BaseModel):

    consulta: str = Field(
        description="Pregunta relacionada con Cambios y Devoluciones de Hiraoka."
    )

    k: int = Field(
        default=5,
        ge=1,
        le=10,
        description="Cantidad máxima de fragmentos a recuperar."
    )

    section_slug: Optional[str] = Field(
        default=None,
        description="""
        Sección específica del documento.

        Ejemplos:
        - politica_cambios_devoluciones_garantias
        - politica_devoluciones
        - productos_digitales
        - derecho_arrepentimiento
        - devolucion_garantia
        - conflicto_solucionado
        """
    )

In [29]:
SECTION_HINTS = {

    # Devoluciones
    "devolucion": "politica_devoluciones",
    "devolución": "politica_devoluciones",
    "devolver": "politica_devoluciones",
    "cambio": "politica_devoluciones",
    "cambiar": "politica_devoluciones",
    "restitucion": "derecho_restitucion",
    "restitución": "derecho_restitucion",
    "reembolso": "devolucion_dinero",
    "extorno": "devolucion_dinero",
    "nota de credito": "devolucion_dinero",
    "nota de crédito": "devolucion_dinero",
    "cheque": "devolucion_dinero",

    # Arrepentimiento
    "arrepentimiento": "derecho_arrepentimiento",
    "ya no lo quiero": "derecho_arrepentimiento",
    "me equivoque": "derecho_arrepentimiento",
    "me equivoqué": "derecho_arrepentimiento",
    "no me gusto": "derecho_arrepentimiento",
    "no me gustó": "derecho_arrepentimiento",
    "devolver sin defecto": "derecho_arrepentimiento",
    "devolver por gusto": "derecho_arrepentimiento",

    # Condiciones
    "producto usado": "productos_usados",
    "producto abierto": "productos_usados",
    "producto desgastado": "productos_usados",

    "dni": "acreditacion_titular",
    "titular": "acreditacion_titular",
    "beneficiario": "acreditacion_titular",
    "poder simple": "acreditacion_titular",

    # Productos digitales
    "software": "productos_digitales",
    "licencia": "productos_digitales",
    "licencias": "productos_digitales",
    "producto digital": "productos_digitales",
    "programa": "productos_digitales",

    # Procedimiento
    "atencion al cliente": "solicitud_devolucion",
    "servicio al cliente": "solicitud_devolucion",
    "como solicitar": "solicitud_devolucion",
    "cómo solicitar": "solicitud_devolucion",
    "reclamo": "solicitud_devolucion",

    # Garantías
    "garantia": "garantias_fabricantes",
    "garantía": "garantias_fabricantes",
    "fabricante": "garantia_fabricante",
    "proveedor": "garantia_fabricante",

    "intermediacion": "intermediacion_hiraoka",
    "intermediación": "intermediacion_hiraoka",

    "garantia fabricante": "garantia_fabricante",
    "garantía fabricante": "garantia_fabricante",

    "plazo garantia": "plazos_garantia",
    "plazo garantía": "plazos_garantia",
    "45 dias": "plazos_garantia",
    "45 días": "plazos_garantia",
    "15 dias": "plazos_garantia",
    "15 días": "plazos_garantia",

    # Productos sensibles
    "producto sensible": "productos_sensibles",
    "productos sensibles": "productos_sensibles",

    "audifonos": "definicion_productos_sensibles",
    "audífonos": "definicion_productos_sensibles",
    "afeitadora": "definicion_productos_sensibles",
    "depiladora": "definicion_productos_sensibles",
    "termometro": "definicion_productos_sensibles",
    "termómetro": "definicion_productos_sensibles",
    "nebulizador": "definicion_productos_sensibles",
    "trotadora": "definicion_productos_sensibles",
    "bicicleta estacionaria": "definicion_productos_sensibles",
    "eliptica": "definicion_productos_sensibles",
    "elíptica": "definicion_productos_sensibles",

    "inspeccion": "inspeccion_productos_sensibles",
    "inspección": "inspeccion_productos_sensibles",
    "evaluacion": "inspeccion_productos_sensibles",
    "evaluación": "inspeccion_productos_sensibles",

    "color": "cambios_por_color_diseno",
    "diseño": "cambios_por_color_diseno",
    "diseno": "cambios_por_color_diseno",
    "modo de uso": "cambios_por_color_diseno",

    # Disposiciones generales
    "responsabilidad": "no_reconocimiento_responsabilidad",
    "monto devolucion": "limite_monto_devolucion",
    "monto devolución": "limite_monto_devolucion",
    "conflicto solucionado": "conflicto_solucionado",
}

In [22]:
def format_results_for_agent(
    results,
    max_chars: int = 1200
):

    if not results:
        return (
            "No se encontró información relevante "
            "Cambios y Devoluciones."
        )

    blocks = []

    for i, (doc, score) in enumerate(results, start=1):

        meta = doc.metadata

        content = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        # Construir título jerárquico de la sección
        sec_title = meta.get('section_title', '')
        if meta.get('subsection_title'):
            sec_title += f" -> {meta.get('subsection_title')}"
        if meta.get('sub_subsection_title'):
            sec_title += f" -> {meta.get('sub_subsection_title')}"

        blocks.append(
            f"""
[Fuente {i}]

Documento: {meta.get('documento')}

Sección: {sec_title}

Slug: {meta.get('sub_subsection_slug') or meta.get('subsection_slug') or meta.get('section_slug')}

Páginas: {meta.get('page_start')} - {meta.get('page_end')}

Chunk ID: {meta.get('chunk_id')}

Score: {score:.4f}

Contenido:
{content}
"""
        )

    return "\n\n".join(blocks)

### Herramienta principal

In [33]:
from langchain.tools import tool


@tool(args_schema=BuscarTerminosHiraokaInput)
def buscar_terminos_hiraoka(
    consulta: str,
    k: int = 5,
    section_slug: Optional[str] = None,
) -> str:
    """
    Busca información dentro de los
    Cambios y Devoluciones de Hiraoka
    utilizando Elasticsearch.
    """

    # Detección automática de sección

    if not section_slug:

        query_lower = consulta.lower()

        for keyword, detected_section in SECTION_HINTS.items():

            if keyword in query_lower:

                section_slug = detected_section
                break

    filters = build_metadata_filter(
        section_slug=section_slug,
        documento= DOCUMENT_TITLE
    )

    results = vector_store.similarity_search_with_score(
        query=consulta,
        k=k,
        filter=filters or None,
    )

    results = [
        r for r in results
        if r[1] >= 0.65
    ]

    results = sorted(
        results,
        key=lambda x: x[1],
        reverse=True
    )

    return format_results_for_agent(results)

In [34]:
print(
    buscar_terminos_hiraoka.invoke(
        {
            "consulta": "¿Cómo hago un cambio?"
        }
    )
)


[Fuente 1]

Documento: Cambios y Devoluciones

Sección: POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS -> I. Política de devoluciones brindadas por Hiraoka

Slug: politica_devoluciones

Páginas: 1 - 1

Chunk ID: politica_devoluciones_0001

Score: 0.7511

Contenido:
I. Política de devoluciones brindadas por Hiraoka  Importaciones Hiraoka S.A.C. (en adelante, “Hiraoka”) brinda a todos sus clientes el derecho de reparación, restitución o devolución, así como los derechos legalmente reconocidos en el Código de Protección y Defensa del Consumidor (en adelante, “Código”), de acuerdo a los términos y condiciones establecidos en el presente documento; siempre y cuando, el cliente cumpla con las siguientes condiciones:  Los productos que hayan sido usados o presenten algún tipo de desgaste, podrán ser cambiados si es que presentan algún defecto, desperfecto y/o problema de idoneidad, atribuible a Hiraoka y/o a los Fabricantes. Por el contrario, si los productos no presentan defecto alguno y se 

In [25]:
print(splits[0].metadata)

{'documento': 'Cambios y Devoluciones', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Cambios y Devoluciones', 'page_start': 1, 'page_end': 1, 'section_slug': 'politica_cambios_devoluciones_garantias', 'section_title': 'POLÍTICA DE CAMBIOS, DEVOLUCIONES Y GARANTIAS', 'chunk_id': 'politica_cambios_devoluciones_garantias_0000', 'chunk_number': 0}


## 2.3 Prueba directa de la herramienta

Antes de conectar la herramienta al agente, conviene probarla de forma aislada.

In [ ]:
test_questions = [

    "¿Cuáles son las condiciones para devolver un producto?",

    "¿Necesito presentar mi DNI?",

    "¿Puede otra persona hacer el trámite por mí?",

    "¿Cómo solicito una devolución?",

]

for question in test_questions:

    print("\n\n")
    print("#" * 120)
    print("PREGUNTA")
    print(question)
    print("#" * 120)

    results = vector_store.similarity_search_with_score(
        question,
        k=3
    )

    print_retrieval_results(
        results,
        max_chars=400
    )




########################################################################################################################
PREGUNTA
¿Cuál es el horario de atención los sábados?
########################################################################################################################

RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8096 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : horario_atencion_0068
Documento     : Terminos y Condiciones
Sección       : Mecanismos de entrega de productos (entrega_productos)
Páginas       : 9 - 10
Autor         : Hiraoka
Año           : 2026
Dominio       : Términos y Condiciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
con la Av. El Bosque) Lunes - Sábado: 10:00 a.m. a  8:00 p

## 2.4 Crear agente básico con LangChain 1.x

El agente tendrá una sola herramienta de recuperación. La instrucción principal obliga a responder con sustento: documento, sección y página.

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    CHAT_MODEL,
    temperature=0
)

SYSTEM_PROMPT = """
Eres un asistente especializado en las Políticas de Cambios,
Devoluciones y Garantías de Hiraoka.

Tu objetivo es responder consultas relacionadas con:

- Cambios de productos
- Devoluciones
- Derecho de arrepentimiento
- Reembolsos
- Extornos
- Garantías
- Productos defectuosos
- Productos sensibles
- Procedimientos de atención al cliente
- Restricciones y condiciones de devolución

INFORMACIÓN FIJA (NO ES NECESARIO CONSULTAR LA HERRAMIENTA PARA ESTOS CASOS)

Datos de contacto:

- Teléfono: (01) 680-3800
- WhatsApp: 969872372
- Correo: servicioalcliente@hiraoka.com.pe

Información de la empresa:

- Razón social: IMPORTACIONES HIRAOKA S.A.C.
- RUC: 20100016681
- Dirección: Av. Abancay 594, Cercado de Lima

REGLAS DE USO DE LA HERRAMIENTA

Utiliza la herramienta buscar_terminos_hiraoka cuando la consulta requiera
información específica de las políticas de cambios, devoluciones o garantías.

Especialmente para temas relacionados con:

- Cambios
- Devoluciones
- Garantías
- Derecho de arrepentimiento
- Reembolsos
- Extornos
- Notas de crédito
- Cheques de devolución
- Productos usados
- Productos abiertos
- Productos digitales
- Licencias y software
- Productos sensibles
- Audífonos
- Afeitadoras
- Depiladoras
- Equipos médicos
- Trotadoras
- Bicicletas estacionarias
- Elípticas
- Procedimientos de evaluación
- Restricciones de devolución
- Plazos de garantía
- Fabricantes o proveedores

INSTRUCCIONES

1. No inventes información.

2. Si la herramienta no devuelve evidencia suficiente, responde:

"No encontré información suficiente en las políticas de cambios, devoluciones y garantías de Hiraoka para responder esta consulta."

3. Prioriza siempre la evidencia recuperada desde la herramienta.

4. Si existen varias políticas aplicables, explica claramente las diferencias.

Por ejemplo:

- Devolución por defecto o falla del producto.
- Devolución por arrepentimiento de compra.
- Garantía del fabricante.

5. Responde siempre en español.

6. Si la consulta está fuera del alcance de las políticas de cambios, devoluciones y garantías, indícalo claramente.

7. Si la consulta es únicamente sobre:

- teléfono
- whatsapp
- correo
- dirección

NO utilices la herramienta.

8. Para consultas relacionadas con:

- garantía
- garantías
- devolución
- devoluciones
- cambio
- cambios
- reembolso
- extorno
- arrepentimiento
- producto defectuoso
- producto usado
- producto abierto
- producto sensible
- audífonos
- afeitadora
- depiladora
- termómetro
- nebulizador
- trotadora
- bicicleta estacionaria
- software
- licencia
- fabricante
- proveedor

utiliza la herramienta para obtener evidencia.

9. Cuando una consulta pueda pertenecer a más de una política,
explica las alternativas aplicables.

FORMATO DE CITAS

Cuando uses información recuperada desde la herramienta, agrega:

Fuentes:
- Documento: <documento>
- Sección: <section_title>
- Páginas: <page_start>-<page_end>

Si la respuesta proviene únicamente de la información fija,
no es necesario mostrar fuentes.
"""

agent = create_agent(
    model=model,
    tools=[buscar_terminos_hiraoka],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
question = "¿Cuánto demora un extorno?""
response = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'No encontré información suficiente en los Términos y Condiciones de Hiraoka para responder esta consulta. La sección de garantías menciona que Hiraoka se rige por su "Política de Cambios y Devoluciones y Garantías", pero no detalla el procedimiento específico para problemas de sistema en productos como un iPhone ni si Hiraoka realiza el servicio técnico directamente o si se debe acudir a un centro autorizado de la marca.', 'extras': {'signature': 'Cu8FAQw51sfI08xLqrkrlGW46h5N6TpjORMRtR6W7WQyT1I9DE0Dzt1oItoU5M82QIbF8Fw/gERmINBkaHByNHnAI3rnHFxIuHbgf+RWcVrKHygruvneEwinmLw5L3xa9oikhiretrGtbFdiarYORs3lQWPUSIhyE181VTOwsXZg5XoiU+EL0HxSi3pzk+E5l1K/wBjoZm9xBNm9yrWI9CW+SC/79HdKTRSZMT3PLGynu74e8j7AoEDRVE+PwLjjxUku48n0WbylXcmCY76D+zYjq1MuiaVSiA6NrBVS7GxHczLP2zlJpyVc6NWFIOGBMPQxpmirXKsCKGqAfkKfzXNBewrePzgcGkORl5xL0hNMcGRXo3JR1RM67RzILyFhsyR5FLBi/W4piAH0eyFueEDAAfseAXvkBMNAWhp7vBom6tXpEPFnikIatmtY7FtO3FX9p3TyGeIxf1tZv4QXTA403qwPj4HKL80Hdryto9wGuJVSMo1y39U82C8Av/YUttwrGA+i/

## 2.5 Preguntas de práctica para el agente

In [ ]:
preguntas_practica = [

    "Si el iPhone qué les he comprado tiene problemas de sistema uds lo ven o tengo que llevarlo a la tienda de iPhone?",

    "¿Cuántos días tengo para recoger un producto comprado por la web?",

    "¿Qué sucede si no recojo mi pedido dentro del plazo establecido?",

    "¿Cómo funciona el servicio de delivery de Hiraoka?",

    "¿Qué distritos están cubiertos por el servicio de entrega?",

    "¿Qué métodos de pago acepta Hiraoka?",

    "¿Cómo funcionan las compras mediante OKA?",

    "¿Qué condiciones aplican a las promociones Arma tu Combo?",

    "¿Qué información brinda el documento sobre garantías de productos?",

    "¿Puede variar el stock mostrado en la página web?",

    "¿Qué ocurre si un producto comprado no se encuentra disponible?",

    "¿Cómo puedo contactar a Hiraoka para consultas o reclamos?",

    "¿Qué situaciones se consideran casos de fuerza mayor?",

    "¿Cómo funciona la reprogramación de entregas?",

    "¿Qué restricciones existen para el recojo por terceros?"
]

In [ ]:
for pregunta in preguntas_practica[:5]:

    print("=" * 120)
    print("PREGUNTA:")
    print(pregunta)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": pregunta
                }
            ]
        }
    )

    print("\nRESPUESTA:")
    print(response["messages"][-1].content)
    print("\n")

PREGUNTA:
Si el iPhone qué les he comprado tiene problemas de sistema uds lo ven o tengo que llevarlo a la tienda de iPhone?

RESPUESTA:
[{'type': 'text', 'text': 'No encontré información suficiente en los Términos y Condiciones de Hiraoka para responder esta consulta. La sección de garantías menciona que Hiraoka se rige por su "Política de Cambios y Devoluciones y Garantías", pero no detalla el procedimiento específico para problemas de sistema en productos como un iPhone ni si la atención la brinda Hiraoka directamente o un servicio técnico autorizado de la marca.', 'extras': {'signature': 'CswFAQw51sejJUO6fc76HhxbUxw8vvsHk8S6iRhTJZEC8y+nu3eNMD/gE/j2yb+S01L5cOD8h7kChDAg2gPCLgz5IKQLuVd0amaVVOcrox7jIOP94oZd2d8cpjsEsL9bvFpZ9aFKyMORfB46RDfB9qcLg1zD8Op3DRtzYV6e5I6zXbiDkVnDLRNpeMCLLkhz5AASyWadq+zCA2PNPxwpNL/cqYqBygdXnZm/K7gvfdEppqUo4fij4QDuo5Op2IJVnCMhUJC3ftdnHrKS5MQ8Iq2G2dyOEIoib8iJy4S/CGlSRa2hnaAbJwNiEw47NOIsLMyF1V1jGKHNJgF4TKRuOvRXcptUCuSP7hm1BxdhucZ8rpUPlZ1tCYcWJa3kGwYAisDJmo/sw01R05BX

KeyboardInterrupt: 

In [ ]:
results = vector_store.similarity_search_with_score(
    "Quiero devolverlo o cambiar por otro celular",
    k=10
)

print_retrieval_results(results)


RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8201 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : reprogramaciones_0080
Documento     : Terminos y Condiciones
Sección       : Reprogramaciones (reprogramaciones)
Páginas       : 15 - 16
Autor         : Hiraoka
Año           : 2026
Dominio       : Términos y Condiciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
entrega de productos no dará lugar a compensación económica, reembolso o responsabilidad adicional por parte de Hiraoka cuando dichos cambios se generen bajo los supuestos previamente mencionados y hayan sido previa y debidamente informados por Hiraoka. Asimismo, los Usuarios aceptan que es su responsabilidad gestionar cualquier solicitud por inconveniente derivado de la r

In [ ]:
print(
    buscar_terminos_hiraoka.invoke(
        {
            "consulta":"Quiero devolverlo o cambiar por otro celular"
        }
    )
)


[Fuente 1]

Documento: Terminos y Condiciones

Sección: Reprogramaciones

Slug: reprogramaciones

Páginas: 15 - 16

Chunk ID: reprogramaciones_0080

Score: 0.8201

Contenido:
entrega de productos no dará lugar a compensación económica, reembolso o responsabilidad adicional por parte de Hiraoka cuando dichos cambios se generen bajo los supuestos previamente mencionados y hayan sido previa y debidamente informados por Hiraoka. Asimismo, los Usuarios aceptan que es su responsabilidad gestionar cualquier solicitud por inconveniente derivado de la reprogramación de entregas. Para ello, el Usuario cuenta con un plazo de hasta siete (7) días naturales a partir de la fecha de notificada la reprogramación de su pedido y deberá de contactarse al número: (01) 680-3800



[Fuente 2]

Documento: Terminos y Condiciones

Sección: Procedimiento de Selección de Productos y Pago -> Medidas de seguridad adoptadas en el procedimiento de pago -> PagoEfectivo

Slug: pago_efectivo

Páginas: 5 - 5

Chunk ID:

# -----------------------